# 03 · 标量自动求导引擎（Value）

> **本节属于 Part 2 · 标量自动求导。这是整个项目的灵魂所在。**

上一节我们手推了反向传播，并深刻体会到：**每换一个模型就要重推梯度**，根本无法扩展。

本节我们造一个 `Value` 类来终结这个痛点。它的核心思想只有一句话：

> **每个数值不仅记住自己的"值"，还记住"自己是怎么被算出来的"。**

有了这条"计算的历史"，我们就能在最后一句 `.backward()` 里，用链式法则**自动**把梯度反传到每一个输入。这正是 PyTorch / TensorFlow 最核心的魔法——而我们要亲手把它造出来。

## 学习目标

- 理解自动求导的本质 = **构建计算图 + 拓扑序上的链式法则**
- 亲手实现 `Value` 类：支持 `+ - * / **`、`tanh / exp / relu`，并能 `.backward()`
- 理解"**局部梯度的闭包 `_backward`**"与"**拓扑排序**"两个关键机制
- 用有限差分与 PyTorch 双重验证

## 直觉与数学原理

把一次计算想象成一张**计算图**：每个 `Value` 是一个节点，每个运算（加、乘、tanh…）把若干输入节点连到一个输出节点。

- **前向**：从输入到输出，正常地把数值算出来——同时，每个运算都"偷偷记下"它的输入是谁、自己是什么运算。
- **反向**：从输出回到输入，对每个节点应用链式法则：

$$\underbrace{\frac{\partial L}{\partial \text{输入}}}_{\text{要传给输入的梯度}} \mathrel{+}= \underbrace{\frac{\partial \text{输出}}{\partial \text{输入}}}_{\text{这一步的局部导数}} \times \underbrace{\frac{\partial L}{\partial \text{输出}}}_{\text{上游梯度}}$$

实现上的两个关键技巧：

1. **每个运算定义一个 `_backward` 闭包**：它知道"如何把输出的梯度分配给自己的输入"。比如乘法 `out = a*b`，则 `a.grad += b.data * out.grad`，`b.grad += a.data * out.grad`。
2. **反向要按正确顺序**：必须先算完一个节点的**所有**上游，才能往它的输入传。这正是**拓扑排序**——先对图做拓扑排序，再**逆序**依次调用每个节点的 `_backward`。

> 注意上面是 `+=`（累加）：如果一个节点被用了多次，它会从多条路径收到梯度，必须累加起来。

## 从零手写实现

下面是完整的 `Value` 类。逐段读注释，你会发现每个运算都遵循同一个套路：**正向算 `data`，并挂上一个把梯度往回传的 `_backward` 闭包**。

In [ ]:
import math

class Value:
    """包装一个标量，并自动构建用于反向传播的计算图。"""

    def __init__(self, data, _children=(), _op=""):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None     # 叶子默认什么都不做
        self._prev = set(_children)        # 直接前驱（图的边）
        self._op = _op                     # 运算名（仅用于可视化）

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        def _backward():
            self.grad += out.grad          # 加法局部导数都是 1
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")
        def _backward():
            self.grad += other.data * out.grad   # 对一个因子求导 = 另一个因子
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "只支持常数次幂"
        out = Value(self.data ** other, (self,), f"**{other}")
        def _backward():
            self.grad += (other * self.data ** (other - 1)) * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Value(0.0 if self.data < 0 else self.data, (self,), "ReLU")
        def _backward():
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), "tanh")
        def _backward():
            self.grad += (1 - t * t) * out.grad   # tanh' = 1 - tanh^2
        out._backward = _backward
        return out

    def exp(self):
        out = Value(math.exp(self.data), (self,), "exp")
        def _backward():
            self.grad += out.data * out.grad       # exp' = exp
        out._backward = _backward
        return out

    def backward(self):
        """按拓扑序反向传播，填充图中所有节点的 grad。"""
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)
        self.grad = 1.0                    # dL/dL = 1
        for v in reversed(topo):           # 从输出往输入
            v._backward()

    # ---- 便捷运算符（都基于上面几个基本运算） ----
    def __neg__(self):        return self * -1
    def __radd__(self, o):    return self + o
    def __sub__(self, o):     return self + (-o)
    def __rsub__(self, o):    return o + (-self)
    def __rmul__(self, o):    return self * o
    def __truediv__(self, o): return self * o ** -1
    def __rtruediv__(self, o):return o * self ** -1
    def __repr__(self):       return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

## 用一下：一个神经元

我们搭一个最小的神经元 $o = \tanh(w_1 x_1 + w_2 x_2 + b)$，前向算出 $o$，再一句 `o.backward()` 自动得到所有梯度。

In [ ]:
x1, x2 = Value(2.0), Value(0.0)
w1, w2 = Value(-3.0), Value(1.0)
b = Value(6.8813735)

n = x1 * w1 + x2 * w2 + b      # 加权和
o = n.tanh()                   # 激活
o.backward()                   # 自动反向传播！

print("o =", o)
print("dO/dx1 =", x1.grad)
print("dO/dw1 =", w1.grad)
print("dO/dx2 =", x2.grad)
print("dO/dw2 =", w2.grad)

## 验证

### 验证一：有限差分

我们用中心差分独立地估计 $\partial o/\partial w_1$，看是否与 `Value` 自动算的一致。

In [ ]:
def neuron_o(w1_val):
    # 用普通 float 重算一遍前向，仅改变 w1
    return math.tanh(2.0 * w1_val + 0.0 * 1.0 + 6.8813735)

eps = 1e-6
fd = (neuron_o(-3.0 + eps) - neuron_o(-3.0 - eps)) / (2 * eps)
print(f"有限差分 dO/dw1 = {fd:.6f}")
print(f"Value 自动   dO/dw1 = {w1.grad:.6f}")
print("一致 ->", abs(fd - w1.grad) < 1e-5)

### 验证二：PyTorch 对照

In [ ]:
import torch

x1t = torch.tensor(2.0); x2t = torch.tensor(0.0)
w1t = torch.tensor(-3.0, requires_grad=True)
w2t = torch.tensor(1.0, requires_grad=True)
bt = torch.tensor(6.8813735, requires_grad=True)
ot = torch.tanh(x1t * w1t + x2t * w2t + bt)
ot.backward()

print(f"PyTorch dO/dw1 = {w1t.grad.item():.6f}   | Value = {w1.grad:.6f}")
print(f"PyTorch dO/dw2 = {w2t.grad.item():.6f}   | Value = {w2.grad:.6f}")

## 看看计算图长什么样

`_op` 和 `_prev` 让我们能把这张图打印出来（这里用最朴素的缩进树，不依赖额外库）。

In [ ]:
def show_graph(v, indent=0, seen=None):
    seen = seen if seen is not None else set()
    label = v._op if v._op else "leaf"
    print("    " * indent + f"[{label}] data={v.data:.4f}, grad={v.grad:.4f}")
    for child in v._prev:
        show_graph(child, indent + 1, seen)

show_graph(o)

## 📦 沉淀进 minitorch

这个 `Value` 类已经整理进 **`minitorch/scalar.py`**。下一节起我们直接从包里导入它，不再重复编写。

> 提示：`Value` 是帮助理解 autograd 的**标量**踏脚石，因此放在 `minitorch.scalar` 而非主命名空间——Part 3 我们会把同样的思想升级到**张量**引擎 `minitorch.Tensor`，那才是后续真正用于训练的核心。

下面从包里导入，复验一遍，确认与我们手写的完全一致：

In [ ]:
from minitorch.scalar import Value as PkgValue

a = PkgValue(2.0); b = PkgValue(-3.0)
L = (a * b + a ** 2).tanh()
L.backward()
print("从包导入的 Value 也能自动求导：")
print("  a.grad =", round(a.grad, 6), " b.grad =", round(b.grad, 6))

## 小练习

1. **加一个运算**：给 `Value` 添加 `sigmoid` 方法（提示：$\sigma(x)=1/(1+e^{-x})$，可用已有的 `exp` 和运算符拼出来），并用有限差分验证它的梯度。
2. **共享节点**：构造表达式 `y = a*a*a`（`a` 用三次），手算 `dy/da = 3a^2`，再用 `Value` 验证——体会 `_backward` 里为什么必须用 `+=` 累加。
3. **画更大的图**：把第 2 题的表达式用 `show_graph` 打印出来，观察 `a` 节点是如何被多条边共享的。

## 小结 & 下一站

✅ 我们亲手造出了自动求导引擎 `Value`——前向时自动建图，`.backward()` 时按拓扑序用链式法则自动反传。**这是本项目最重要的一块基石。**

✅ 关键机制：每个运算挂一个 `_backward` 闭包；反向按拓扑逆序执行；梯度用 `+=` 累加。

**下一站 → `04_neuron_mlp_with_value`**：我们用 `Value` 搭出 `Neuron / Layer / MLP`，真正训练一个二分类模型，并借此看清一个问题——**标量引擎太慢了**，从而引出 Part 3 的张量引擎。